# 02 — The Rust Laya daemon surface (`laya-jsonl`)

**Question this notebook answers:** what exactly crosses the boundary
between the controller and the System One decision model?

Laya never generates text. You send a **state** and **typed questions**;
it returns typed answers with calibrated probabilities in one forward
pass. The `laya-jsonl` daemon keeps the checkpoint loaded and speaks
JSON-lines on stdin/stdout. Companion reference:
`docs/INTERACTION_MAP.md` §2.


In [1]:
import os, sys

# make the `lab` package importable whether jupyter starts here or in the repo root
for _p in (os.getcwd(), os.path.dirname(os.getcwd())):
    if os.path.exists(os.path.join(_p, "lab")):
        sys.path.insert(0, _p)
        PROJ = _p
        break
else:
    raise RuntimeError("run jupyter from the project root or the notebooks/ directory")

from lab import EwmScene, LayaDaemon, BonsaiClient
print("project root:", PROJ)



project root: /home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/ewm-laya-bonsai-lab


In [2]:
laya = LayaDaemon()
print("daemon binary:", laya.bin)
print("checkpoint:", laya.model_dir)


daemon binary: /home/alexmy/tools/laya-rust/target/release/laya-jsonl
checkpoint: /home/alexmy/.cache/laya/typed-decisions


## A raw `choice` request

`criteria` is an object `{name: description}`. The answer comes back as a
distribution over the **names**. Notice the response answers are sorted
alphabetically by question name (the daemon uses a `BTreeMap`).


In [3]:
resp = laya.request({
    "state": "The user asked: \"What is the capital of France?\". The system's memory holds 0 token ids.",
    "questions": {
        "route": {
            "type": "choice",
            "instructions": "How should the next context be built for the base model?",
            "criteria": {
                "full": "keep the full materialized memory",
                "compact": "keep only new, previously unseen tids",
            },
        },
    },
}, verbose=True)


── laya request ──
{
  "id": 1,
  "state": "The user asked: \"What is the capital of France?\". The system's memory holds 0 token ids.",
  "questions": {
    "route": {
      "type": "choice",
      "instructions": "How should the next context be built for the base model?",
      "criteria": {
        "full": "keep the full materialized memory",
        "compact": "keep only new, previously unseen tids"
      }
    }
  }
}
── laya response ──
{
  "id": 1,
  "response": {
    "model": "laya-typed-decisions",
    "answers": {
      "route": {
        "type": "choice",
        "choice": "full",
        "probabilities": {
          "full": 0.5101,
          "compact": 0.4899
        },
        "confidence": 0.0003,
        "rl_agent": {
          "act_probability": 1.0
        }
      }
    },
    "usage": {
      "input_tokens": 61,
      "output_tokens": 0
    }
  }
}


## A raw `noul` request

`noul` = "yes/no, does this statement hold". The answer is P(holds), a
single number between 0 and 1.


In [4]:
laya.request({
    "state": "The user asked: \"Explain gravity in one sentence.\". The memory holds 24 tids.",
    "questions": {"keep_short": {"type": "noul", "instructions": "The context should be kept short"}},
}, verbose=True)


── laya request ──
{
  "id": 2,
  "state": "The user asked: \"Explain gravity in one sentence.\". The memory holds 24 tids.",
  "questions": {
    "keep_short": {
      "type": "noul",
      "instructions": "The context should be kept short"
    }
  }
}
── laya response ──
{
  "id": 2,
  "response": {
    "model": "laya-typed-decisions",
    "answers": {
      "keep_short": {
        "type": "noul",
        "noul": 0.4126,
        "rl_agent": {
          "act_probability": 1.0
        }
      }
    },
    "usage": {
      "input_tokens": 52,
      "output_tokens": 0
    }
  }
}


{'id': 2,
 'response': {'model': 'laya-typed-decisions',
  'answers': {'keep_short': {'type': 'noul',
    'noul': 0.4126,
    'rl_agent': {'act_probability': 1.0}}},
  'usage': {'input_tokens': 52, 'output_tokens': 0}}}

## A raw `score` request

`score` is ordinal: the answer is the expectation over level indices,
with a legend and per-level probabilities.


In [5]:
laya.request({
    "state": "The user asked: \"Write a haiku about rain.\". The memory holds 40 tids.",
    "questions": {
        "urgency": {
            "type": "score",
            "instructions": "How much context does this query need?",
            "criteria": ["none", "a little", "a lot"],
        },
    },
}, verbose=True)


── laya request ──
{
  "id": 3,
  "state": "The user asked: \"Write a haiku about rain.\". The memory holds 40 tids.",
  "questions": {
    "urgency": {
      "type": "score",
      "instructions": "How much context does this query need?",
      "criteria": [
        "none",
        "a little",
        "a lot"
      ]
    }
  }
}
── laya response ──
{
  "id": 3,
  "response": {
    "model": "laya-typed-decisions",
    "answers": {
      "urgency": {
        "type": "score",
        "score": 1.0598,
        "legend": {
          "0": "none",
          "1": "a little",
          "2": "a lot"
        },
        "probabilities": {
          "0": 0.257,
          "1": 0.4263,
          "2": 0.3168
        },
        "confidence": 0.0199,
        "rl_agent": {
          "act_probability": 1.0
        }
      }
    },
    "usage": {
      "input_tokens": 52,
      "output_tokens": 0
    }
  }
}


{'id': 3,
 'response': {'model': 'laya-typed-decisions',
  'answers': {'urgency': {'type': 'score',
    'score': 1.0598,
    'legend': {'0': 'none', '1': 'a little', '2': 'a lot'},
    'probabilities': {'0': 0.257, '1': 0.4263, '2': 0.3168},
    'confidence': 0.0199,
    'rl_agent': {'act_probability': 1.0}}},
  'usage': {'input_tokens': 52, 'output_tokens': 0}}}

## Batching — one forward pass for all questions

All questions attached to one state go through a single batched forward
pass, so asking five questions costs about as much as asking one.


In [6]:
laya.request({
    "state": "The user asked: \"Name three planets in our solar system.\". The memory holds 12 tids.",
    "questions": {
        "route": {
            "type": "choice",
            "instructions": "Which context mode fits this query?",
            "criteria": {"full": "whole memory", "compact": "new tids only", "surprise": "surprising frames only"},
        },
        "keep_short": {"type": "noul", "instructions": "The context should be kept short"},
        "needs_reasoning": {"type": "noul", "instructions": "This query needs a reasoning model"},
    },
}, verbose=True)


── laya request ──
{
  "id": 4,
  "state": "The user asked: \"Name three planets in our solar system.\". The memory holds 12 tids.",
  "questions": {
    "route": {
      "type": "choice",
      "instructions": "Which context mode fits this query?",
      "criteria": {
        "full": "whole memory",
        "compact": "new tids only",
        "surprise": "surprising frames only"
      }
    },
    "keep_short": {
      "type": "noul",
      "instructions": "The context should be kept short"
    },
    "needs_reasoning": {
      "type": "noul",
      "instructions": "This query needs a reasoning model"
    }
  }
}
── laya response ──
{
  "id": 4,
  "response": {
    "model": "laya-typed-decisions",
    "answers": {
      "keep_short": {
        "type": "noul",
        "noul": 0.2419,
        "rl_agent": {
          "act_probability": 1.0
        }
      },
      "needs_reasoning": {
        "type": "noul",
        "noul": 0.194,
        "rl_agent": {
          "act_probability": 1.0
  

{'id': 4,
 'response': {'model': 'laya-typed-decisions',
  'answers': {'keep_short': {'type': 'noul',
    'noul': 0.2419,
    'rl_agent': {'act_probability': 1.0}},
   'needs_reasoning': {'type': 'noul',
    'noul': 0.194,
    'rl_agent': {'act_probability': 1.0}},
   'route': {'type': 'choice',
    'choice': 'full',
    'probabilities': {'full': 0.4936, 'compact': 0.3373, 'surprise': 0.169},
    'confidence': 0.0756,
    'rl_agent': {'act_probability': 1.0}}},
  'usage': {'input_tokens': 159, 'output_tokens': 0}}}

## What the model actually sees

Each question is rendered as

```text
[CLS] <type> question: <instructions> [SEP] [MASK] opt0 [MASK] opt1 ... [SEP] <state> [SEP]
```

Budgets from `rl_agent_config.json`: whole sequence `max_len=1024`,
instructions+options `head_max_len=256`, each option truncated to 48
tokens, the state truncated to the remaining room. `usage.input_tokens`
reports the rendered sequence length; `output_tokens` is always 0 (no
generation). Calibration temperatures are per (type, option-count)
bucket.


## The prose-state convention

The fine-tuned `laya-typed-decisions` checkpoint rewards natural-language
states, not raw structs. The controller renders trajectory features as
prose before asking:


In [7]:
print(LayaDaemon.prose_state(
    "What is the capital of France?",
    memory_tokens=24,
    bss={"llm_a": 0.55, "llm_b": 0.32, "llm_c": 0.21},
))


The user asked: "What is the capital of France?". The system's memory holds 24 content-addressed token ids from earlier answers. The models' current coverage of the conversation is: llm_a 0.550, llm_b 0.320, llm_c 0.210.


In [8]:
laya.close()
print('daemon closed')


daemon closed
